Trong file này chứa:
- **Trong cái này thì hiện tại ban đầu chưa có tiền xử lý**
- Chia tập train và test
- Scale bằng min max
- Smote data
- Chạy và đánh giá với tập test mất cân bằng của mô hình có smote và không có smote
+ Hiện tại đang thấy mô hình base chiến thắng do dữ liệu test nó cũng chỉ nghiên về lớp 1 nhiều nên hầu hết nó đúng.
- Gan cho data(base data) ở đây chỉ gan cho tập train
- Train với mô hình
- Tiếp tục gan cho tập test để cân bằng cho các lớp để xem kết quả
- Kết quả mô hình có gan có tỉ lệ cao hơn mô hình train với dữ liệu cơ bản do
+ Mô hình train với dữ liệu cơ bản nên bị mất cân bằng với các lớp 1(lớp mà chiếm đa số) nên nó sẽ dự đoán sai nhiều với các lớp khác nên tỉ lệ thấp.
+ Mô hình train với dữ liệu Gan do đã có GAN cân bằng rồi nên khả năng dự đoán với các lớp khác sẽ tốt hơn.

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/DATN/fetal_health.csv'
data = pd.read_csv(file_path)
display(data.head())

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability  

In [ ]:
data.shape

(2126, 22)

In [ ]:
# data.duplicated().sum()

In [ ]:
# data.drop_duplicates(inplace=True)

In [ ]:
# data.duplicated().sum()

In [ ]:
# data.isna().sum().sum()

In [ ]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
baseline value,2126.0,133.303857,9.840844,106.0,126.000,133.000,140.000,160.000
accelerations,2126.0,0.003178,0.003866,0.0,0.000,0.002,0.006,0.019
fetal_movement,2126.0,0.009481,0.046666,0.0,0.000,0.000,0.003,0.481
uterine_contractions,2126.0,0.004366,0.002946,0.0,0.002,0.004,0.007,0.015
light_decelerations,2126.0,0.001889,0.002960,0.0,0.000,0.000,0.003,0.015
severe_decelerations,2126.0,0.000003,0.000057,0.0,0.000,0.000,0.000,0.001
prolongued_decelerations,2126.0,0.000159,0.000590,0.0,0.000,0.000,0.000,0.005
abnormal_short_term_variability,2126.0,46.990122,17.192814,12.0,32.000,49.000,61.000,87.000
mean_value_of_short_term_variability,2126.0,1.332785,0.883241,0.2,0.700,1.200,1.700,7.000
percentage_of_time_with_abnormal_long_term_variability,2126.0,9.846660,18.396880,0.0,0.000,0.000,11.000,91.000


In [ ]:
negative_values_found = False
for column in data.select_dtypes(include=['number']).columns:
    negative_count = (data[column] < 0).sum()
    if negative_count > 0:
        print(f"Cột '{column}' có {negative_count} giá trị nhỏ hơn 0.")
        negative_values_found = True

if not negative_values_found:
    print("Không tìm thấy giá trị nào nhỏ hơn 0 trong các cột số.")

Cột 'histogram_tendency' có 165 giá trị nhỏ hơn 0.


In [ ]:
data["histogram_tendency"].value_counts()

,count
histogram_tendency,
0.0,1115
1.0,846
-1.0,165


In [ ]:
data["fetal_health"].value_counts()

,count
fetal_health,
1.0,1655
2.0,295
3.0,176


In [ ]:
from sklearn.model_selection import train_test_split
X = data.drop('fetal_health', axis=1)
y = data['fetal_health']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Kích thước tập huấn luyện X: {X_train.shape}")
print(f"Kích thước tập kiểm tra X: {X_test.shape}")
print(f"Kích thước tập huấn luyện y: {y_train.shape}")
print(f"Kích thước tập kiểm tra y: {y_test.shape}")

Kích thước tập huấn luyện X: (1700, 21)
Kích thước tập kiểm tra X: (426, 21)
Kích thước tập huấn luyện y: (1700,)
Kích thước tập kiểm tra y: (426,)


In [ ]:
# from sklearn.preprocessing import StandardScaler

# # Khởi tạo scaler
# scaler = StandardScaler()

# # Fit trên tập train, transform cả train và test
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # Nếu muốn giữ dạng DataFrame để dễ nhìn tên cột
# X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
# X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# print("Đã scale dữ liệu xong")
# display(X_train_scaled.head())

Scale data

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Khởi tạo scaler
scaler = MinMaxScaler()

# Fit trên train, transform cả train và test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Nếu muốn giữ DataFrame
import pandas as pd
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
print("Đã scale dữ liệu xong")
display(X_train_scaled.head())

Đã scale dữ liệu xong


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
1953,0.500000,0.000000,0.000000,0.857143,0.066667,0.0,0.4,0.648649,0.411765,0.000000,...,0.543353,0.073394,0.284483,0.222222,0.0,0.511811,0.211009,0.256881,0.311024,0.5
1439,0.740741,0.333333,0.000000,0.214286,0.000000,0.0,0.0,0.351351,0.117647,0.000000,...,0.265896,0.697248,0.456897,0.166667,0.0,0.708661,0.724771,0.697248,0.019685,0.5
2033,0.425926,0.000000,0.002096,0.500000,0.400000,0.0,0.4,0.743243,0.441176,0.000000,...,0.445087,0.146789,0.206897,0.388889,0.0,0.354331,0.064220,0.275229,0.035433,0.5
1731,0.518519,0.444444,0.002096,0.714286,0.400000,0.0,0.0,0.662162,0.132353,0.000000,...,0.612717,0.275229,0.577586,0.555556,0.0,0.755906,0.651376,0.678899,0.240157,0.5
241,0.351852,0.000000,0.010482,0.071429,0.066667,0.0,0.0,0.783784,0.029412,0.318681,...,0.450867,0.018349,0.094828,0.277778,0.0,0.511811,0.458716,0.440367,0.007874,1.0


Smote

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# Kiểm tra phân bố lớp trước SMOTE
print("\nTrước SMOTE:")
print(Counter(y_train))

# SMOTE chỉ áp dụng trên train
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Chuyển lại thành DataFrame/Series
X_train_smote = pd.DataFrame(X_train_smote, columns=X_train_scaled.columns)
y_train_smote = pd.Series(y_train_smote, name=y_train.name if hasattr(y_train, 'name') else 'target')

# Kiểm tra sau SMOTE
print("\nSau SMOTE:")
print(Counter(y_train_smote))

display(X_train_smote.head())
display(X_train_smote.shape)
display(y_train_smote.head())


Trước SMOTE:
Counter({1.0: 1323, 2.0: 236, 3.0: 141})

Sau SMOTE:
Counter({3.0: 1323, 1.0: 1323, 2.0: 1323})


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency
0,0.500000,0.000000,0.000000,0.857143,0.066667,0.0,0.4,0.648649,0.411765,0.000000,...,0.543353,0.073394,0.284483,0.222222,0.0,0.511811,0.211009,0.256881,0.311024,0.5
1,0.740741,0.333333,0.000000,0.214286,0.000000,0.0,0.0,0.351351,0.117647,0.000000,...,0.265896,0.697248,0.456897,0.166667,0.0,0.708661,0.724771,0.697248,0.019685,0.5
2,0.425926,0.000000,0.002096,0.500000,0.400000,0.0,0.4,0.743243,0.441176,0.000000,...,0.445087,0.146789,0.206897,0.388889,0.0,0.354331,0.064220,0.275229,0.035433,0.5
3,0.518519,0.444444,0.002096,0.714286,0.400000,0.0,0.0,0.662162,0.132353,0.000000,...,0.612717,0.275229,0.577586,0.555556,0.0,0.755906,0.651376,0.678899,0.240157,0.5
4,0.351852,0.000000,0.010482,0.071429,0.066667,0.0,0.0,0.783784,0.029412,0.318681,...,0.450867,0.018349,0.094828,0.277778,0.0,0.511811,0.458716,0.440367,0.007874,1.0


(3969, 21)

,fetal_health
0,3.0
1,1.0
2,3.0
3,1.0
4,2.0


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Khởi tạo model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model với dữ liệu đã scale
rf_model.fit(X_train_scaled, y_train)

# Predict
y_pred = rf_model.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9272300469483568

Classification Report:
               precision    recall  f1-score   support

         1.0       0.95      0.98      0.96       332
         2.0       0.83      0.68      0.75        59
         3.0       0.86      0.86      0.86        35

    accuracy                           0.93       426
   macro avg       0.88      0.84      0.86       426
weighted avg       0.92      0.93      0.92       426


Confusion Matrix:
 [[325   5   2]
 [ 16  40   3]
 [  2   3  30]]


Mô hình có smote


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Khởi tạo model
rf_smote_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Train model với dữ liệu đã scale
rf_smote_model.fit(X_train_smote, y_train_smote)

# Predict
y_pred_smote = rf_smote_model.predict(X_test_scaled)

# Đánh giá
print("Accuracy:", accuracy_score(y_test, y_pred_smote))
print("\nClassification Report:\n", classification_report(y_test, y_pred_smote))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_smote))

Accuracy: 0.9248826291079812

Classification Report:
               precision    recall  f1-score   support

         1.0       0.96      0.96      0.96       332
         2.0       0.77      0.75      0.76        59
         3.0       0.82      0.89      0.85        35

    accuracy                           0.92       426
   macro avg       0.85      0.86      0.86       426
weighted avg       0.93      0.92      0.92       426


Confusion Matrix:
 [[319  11   2]
 [ 10  44   5]
 [  2   2  31]]


Gan

In [ ]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from collections import Counter
from ctgan import CTGAN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =============================
# 1. Chuẩn hóa kiểu dữ liệu đầu vào
# =============================

# Nếu X_train_scaled, X_test_scaled chưa phải DataFrame thì chuyển sang DataFrame
if not isinstance(X_train_scaled, pd.DataFrame):
    X_train_scaled = pd.DataFrame(X_train_scaled)

if not isinstance(X_test_scaled, pd.DataFrame):
    X_test_scaled = pd.DataFrame(X_test_scaled)

# Nếu y_train, y_test chưa phải Series thì chuyển sang Series
if not isinstance(y_train, pd.Series):
    y_train = pd.Series(y_train, name='fetal_health')
else:
    y_train = y_train.copy()
    y_train.name = 'fetal_health'

if not isinstance(y_test, pd.Series):
    y_test = pd.Series(y_test, name='fetal_health')
else:
    y_test = y_test.copy()
    y_test.name = 'fetal_health'

# Reset index để tránh lệch dòng khi concat
X_train_scaled = X_train_scaled.reset_index(drop=True)
X_test_scaled = X_test_scaled.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# =============================
# 2. Tạo DataFrame train cho GAN
# =============================

target_col = 'fetal_health'

train_gan_df = pd.concat([X_train_scaled, y_train], axis=1)

# CTGAN xử lý cột phân loại tốt hơn khi để dạng string
train_gan_df[target_col] = train_gan_df[target_col].astype(str)

print("Phân bố lớp trước GAN:")
print(train_gan_df[target_col].value_counts())

Phân bố lớp trước GAN:
fetal_health
1.0    1323
2.0     236
3.0     141
Name: count, dtype: int64


In [ ]:
# =============================
# 3. Train CTGAN
# =============================

ctgan = CTGAN(
    epochs=300,
    batch_size=100,
    verbose=True
)

ctgan.fit(train_gan_df, discrete_columns=[target_col])

print("Đã train xong CTGAN")

Gen. (-00.89) | Discrim. (-00.30): 100%|██████████| 300/300 [04:54<00:00,  1.02it/s]

Đã train xong CTGAN


In [ ]:
# import joblib
# ctgan.save('/content/drive/MyDrive/DATN/ctgan_model.pkl')

# print("Đã lưu CTGAN")

In [ ]:
# =============================
# 4. Sinh dữ liệu cho các lớp thiểu số
# =============================

class_counts = train_gan_df[target_col].value_counts()
max_count = class_counts.max()

synthetic_parts = []

for cls, count in class_counts.items():
    need = max_count - count

    if need <= 0:
        continue

    print(f"\nLớp {cls} cần sinh thêm {need} mẫu")

    collected = []
    total_collected = 0

    while total_collected < need:
        sample_n = max(500, need * 2)
        fake_batch = ctgan.sample(sample_n)

        fake_cls = fake_batch[fake_batch[target_col] == cls].copy()

        if len(fake_cls) > 0:
            collected.append(fake_cls)
            total_collected += len(fake_cls)
            print(f"Đã lấy được {total_collected}/{need}")

    fake_cls_final = pd.concat(collected, axis=0).iloc[:need].copy()
    synthetic_parts.append(fake_cls_final)

# Gộp dữ liệu thật + dữ liệu GAN
if len(synthetic_parts) > 0:
    synthetic_df = pd.concat(synthetic_parts, axis=0).reset_index(drop=True)
    train_balanced_df = pd.concat([train_gan_df, synthetic_df], axis=0).reset_index(drop=True)
else:
    train_balanced_df = train_gan_df.copy()

print("\nPhân bố lớp sau GAN:")
print(train_balanced_df[target_col].value_counts())


Lớp 2.0 cần sinh thêm 1087 mẫu
Đã lấy được 677/1087
Đã lấy được 1322/1087

Lớp 3.0 cần sinh thêm 1182 mẫu
Đã lấy được 681/1182
Đã lấy được 1352/1182

Phân bố lớp sau GAN:
fetal_health
3.0    1323
1.0    1323
2.0    1323
Name: count, dtype: int64


In [ ]:
# =============================
# 5. Tách lại X_train_gan và y_train_gan
# =============================

X_train_gan = train_balanced_df.drop(columns=[target_col]).copy()
y_train_gan = train_balanced_df[target_col].astype(float)  # fetal_health thường là 1.0, 2.0, 3.0

print("Kích thước dữ liệu sau GAN:")
print("X_train_gan:", X_train_gan.shape)
print("y_train_gan:", y_train_gan.shape)

Kích thước dữ liệu sau GAN:
X_train_gan: (3969, 21)
y_train_gan: (3969,)


In [ ]:
# =============================
# 6. Train RandomForest
# =============================

rf_gan_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_gan_model.fit(X_train_gan, y_train_gan)

y_gan_pred = rf_gan_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_gan_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_gan_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_gan_pred))

Accuracy: 0.9295774647887324

Classification Report:

              precision    recall  f1-score   support

         1.0       0.96      0.98      0.97       332
         2.0       0.80      0.75      0.77        59
         3.0       0.85      0.80      0.82        35

    accuracy                           0.93       426
   macro avg       0.87      0.84      0.85       426
weighted avg       0.93      0.93      0.93       426


Confusion Matrix:

[[324   6   2]
 [ 12  44   3]
 [  2   5  28]]


ct gan test

In [ ]:
import pandas as pd
from ctgan import CTGAN

# =============================
# 1. Chuẩn hóa kiểu dữ liệu cho TEST
# =============================
target_col = 'fetal_health'

if not isinstance(X_test_scaled, pd.DataFrame):
    X_test_scaled = pd.DataFrame(X_test_scaled)

if not isinstance(y_test, pd.Series):
    y_test = pd.Series(y_test, name=target_col)
else:
    y_test = y_test.copy()
    y_test.name = target_col

X_test_scaled = X_test_scaled.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# =============================
# 2. Tạo DataFrame test cho CTGAN
# =============================
test_gan_df = pd.concat([X_test_scaled, y_test], axis=1)

# ép target sang string để CTGAN xử lý như cột phân loại
test_gan_df[target_col] = test_gan_df[target_col].astype(str)

print("Phân bố lớp trước GAN ở TEST:")
print(test_gan_df[target_col].value_counts())

# =============================
# 3. Train CTGAN riêng cho TEST
# =============================
ctgan_test = CTGAN(
    epochs=300,
    batch_size=100,
    verbose=True
)

ctgan_test.fit(test_gan_df, discrete_columns=[target_col])

print("Đã train xong CTGAN cho TEST")

# =============================
# 4. Sinh thêm mẫu để cân bằng TEST
# =============================
class_counts_test = test_gan_df[target_col].value_counts()
max_count_test = class_counts_test.max()

synthetic_test_parts = []

for cls, count in class_counts_test.items():
    need = max_count_test - count

    if need <= 0:
        continue

    print(f"\nLớp {cls} cần sinh thêm {need} mẫu cho TEST")

    collected = []
    total_collected = 0

    while total_collected < need:
        sample_n = max(500, need * 2)
        fake_batch = ctgan_test.sample(sample_n)

        # đảm bảo cùng kiểu dữ liệu khi lọc
        fake_batch[target_col] = fake_batch[target_col].astype(str)

        fake_cls = fake_batch[fake_batch[target_col] == str(cls)].copy()

        if len(fake_cls) > 0:
            collected.append(fake_cls)
            total_collected += len(fake_cls)
            print(f"Đã lấy được {total_collected}/{need}")

    fake_cls_final = pd.concat(collected, axis=0).iloc[:need].copy()
    synthetic_test_parts.append(fake_cls_final)

# =============================
# 5. Gộp dữ liệu thật + dữ liệu GAN cho TEST
# =============================
if len(synthetic_test_parts) > 0:
    synthetic_test_df = pd.concat(synthetic_test_parts, axis=0).reset_index(drop=True)
    test_balanced_df = pd.concat([test_gan_df, synthetic_test_df], axis=0).reset_index(drop=True)
else:
    test_balanced_df = test_gan_df.copy()

print("\nPhân bố lớp sau GAN ở TEST:")
print(test_balanced_df[target_col].value_counts())

# =============================
# 6. Tách lại X_test_ctgan / y_test_ctgan
# =============================
X_test_ctgan = test_balanced_df.drop(columns=[target_col]).copy()

# đổi target về kiểu số nguyên nếu nhãn của bạn là 1,2,3
y_test_ctgan = pd.to_numeric(test_balanced_df[target_col], errors='coerce').astype(int)

# giữ lại tên cột giống X_test_scaled gốc
X_test_ctgan.columns = X_test_scaled.columns

print("\nKích thước X_test_ctgan:", X_test_ctgan.shape)
print("Kích thước y_test_ctgan:", y_test_ctgan.shape)
print("\nPhân bố y_test_ctgan:")
print(y_test_ctgan.value_counts().sort_index())

Phân bố lớp trước GAN ở TEST:
fetal_health
1.0    332
2.0     59
3.0     35
Name: count, dtype: int64


Gen. (-01.60) | Discrim. (-00.32): 100%|██████████| 300/300 [00:56<00:00,  5.32it/s]


Đã train xong CTGAN cho TEST

Lớp 2.0 cần sinh thêm 273 mẫu cho TEST
Đã lấy được 169/273
Đã lấy được 312/273

Lớp 3.0 cần sinh thêm 297 mẫu cho TEST
Đã lấy được 145/297
Đã lấy được 294/297
Đã lấy được 443/297

Phân bố lớp sau GAN ở TEST:
fetal_health
1.0    332
2.0    332
3.0    332
Name: count, dtype: int64

Kích thước X_test_ctgan: (996, 21)
Kích thước y_test_ctgan: (996,)

Phân bố y_test_ctgan:
fetal_health
1    332
2    332
3    332
Name: count, dtype: int64


thường

In [ ]:

# Predict
y_pred_testctgan = rf_model.predict(X_test_ctgan)

# Đánh giá
print("Accuracy:", accuracy_score(y_test_ctgan, y_pred_testctgan))
print("\nClassification Report:\n", classification_report(y_test_ctgan, y_pred_testctgan))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_ctgan, y_pred_testctgan))

Accuracy: 0.5120481927710844

Classification Report:
               precision    recall  f1-score   support

           1       0.43      0.98      0.60       332
           2       0.79      0.42      0.55       332
           3       0.67      0.14      0.23       332

    accuracy                           0.51       996
   macro avg       0.63      0.51      0.46       996
weighted avg       0.63      0.51      0.46       996


Confusion Matrix:
 [[325   5   2]
 [172 139  21]
 [253  33  46]]


vip pro có ctgan

In [ ]:
# Predict
y_pred_testctgan_sss = rf_gan_model.predict(X_test_ctgan)

# Đánh giá
print("Accuracy:", accuracy_score(y_test_ctgan, y_pred_testctgan_sss))
print("\nClassification Report:\n", classification_report(y_test_ctgan, y_pred_testctgan_sss))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_ctgan, y_pred_testctgan_sss))

Accuracy: 0.7730923694779116

Classification Report:
               precision    recall  f1-score   support

           1       0.96      0.98      0.97       332
           2       0.61      0.90      0.73       332
           3       0.86      0.45      0.59       332

    accuracy                           0.77       996
   macro avg       0.81      0.77      0.76       996
weighted avg       0.81      0.77      0.76       996


Confusion Matrix:
 [[324   6   2]
 [ 12 298  22]
 [  2 182 148]]
